# selenium으로 동적웹페이지 데이터 수집하기_googleplay_금융앱 5개 리뷰수집

In [2]:
!pip install selenium webdriver-manager

# 토스 앱 리뷰 수집하기_최종

In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from datetime import datetime, timedelta
import pandas as pd
import time
from dbio import to_db

# selenium으로 웹브라우저 켜고 접속하는 함수

In [4]:
def create_driver(url):
    # Chrome 옵션 설정
    options = Options()
    options.add_argument("--window-size=1280,900")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    # ChromeDriver 설정
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(30)

    try:
        driver.get(f"https://play.google.com/store/apps/details?id={url}")
        print(f"{url} 접속 성공", end="\r")
    except Exception as e:
        print(e)
        print("driver 생성 실패")
            
    return driver

# 리뷰창 열고 최신순으로 정렬 후 지정한 날짜까지 스크롤하기

In [5]:
def get_reviews(driver, days):
    wait = WebDriverWait(driver, 10)

    # 자바스트립트로 윈도우를 0-1200px까지 스크롤 내리기
    driver.execute_script("window.scrollTo(0, 1200)")
    # 평점 및 리뷰 옆의 -> 버튼 찾아서 클릭하기
    btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[aria-label="평점 및 리뷰 자세히 알아보기"]')))
    btn.click()
    # 최신순으로 리뷰를 정렬하기 위해 버튼 클릭
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#sortBy_1")))
    button.click()
    time.sleep(5)
    # 최신을 찾아 클릭
    button2 = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'span[aria-label="최신"]')))
    button2.click()
    time.sleep(5)
    ###########################################################
    # 팝업된 리뷰창에서 데이터 수집하기
    ###########################################################

    # 오늘 날짜를 추출하고 1달 전 날짜를 계산해 수집할 날짜 범위 정하기
    today = datetime.today()
    end_date = today - timedelta(days=days)

    while True:

        # 리뷰창 스크롤 내려서 과거 리뷰 로딩하기
        driver.execute_script("document.querySelector('.fysCi.Vk3ZVd').scrollBy(0, 1000)")
        time.sleep(1)
        # 리뷰 가장 마지막의 날짜 추출하기
        last_date = driver.find_elements(By.CSS_SELECTOR, ".bp9Aid")[-1].text
        last_date = last_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
        last_date = datetime.strptime(last_date, "%Y-%m-%d")
        n_reviews = len(driver.find_elements(By.CSS_SELECTOR, ".bp9Aid"))
        print("리뷰개수:", n_reviews, "end_date", end_date, "last_date", last_date, end="\r")
        if last_date < end_date:
            break
    return driver.find_elements(By.CSS_SELECTOR, ".RHo1pe")

In [6]:
# 데이터 수집 및 DB 저장 함수
def review_extractnsave(items, url):
    all_reviews = []    
    # 스크롤이 멈춘 후 데이터 수집하기
    for item in driver.find_elements(By.CSS_SELECTOR, ".RHo1pe"):
        result = {} 
        # 리뷰에서 날짜 추출하기
        date = item.find_element(By.CSS_SELECTOR, ".bp9Aid").text
        date = date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
        review_date = datetime.strptime(date, "%Y-%m-%d")
        # 평점 추출하기
        rating = item.find_element(By.CSS_SELECTOR, ".iXRFPc").get_attribute("aria-label")
        rating = rating.split()[3][0]
        # 리뷰 글 추출하기
        review_text = item.find_element(By.CSS_SELECTOR, ".h3YV2d").text

        result["date"] = review_date
        result['rating'] = rating
        result['review_text'] = review_text
        all_reviews.append(result)

    df = pd.DataFrame(all_reviews)
    display(df) 
    to_db("bank_app_reviews", f"{url}_reviews", df)

In [7]:
urls = ["viva.republica.toss", "com.kebhana.hanapush", "com.kbstar.kbbank", "com.shinhan.sbanking", "com.wooribank.smart.npib"]
for url in urls:
    driver = create_driver(url)
    items = get_reviews(driver, 10)
    review_extractnsave(items, url)

,date,rating,review_text
0,2026-03-05,1,용량 봐라...내 폰 자원 1기가 육박하게 잡아먹고 있네 ㅎ;;
1,2026-03-05,1,접속이 안됩니다. 특히 3일전부터 아예 안들어가지는 시간대가 너무 길고 잦아요. 다...
2,2026-03-05,1,업데이트하고 나서 왜 이체할 때 비밀번호 치는 창이 투명하게 뜨는 겁니까? 투명하게...
3,2026-03-05,3,폰트가 뭉게져서 나와요
4,2026-03-04,2,진짜 토스좋아하는데 요즘 에러도 자주나고 느려지고 아주 불편해요ㅜㅜㅜ
...,...,...,...
215,2026-02-24,1,1. 계좌 불러오기 하는데 새마을금고가 없어 여러번 추가 요청하기 했는데도 거의 한...
216,2026-02-24,5,토스 짱
217,2026-02-24,3,잔액 업데이트가 정말 너무 느립니다. 며칠이 지나도 잘못 나오는 건 이제 기본이네요.
218,2026-02-24,1,주식 하고 있는데 왜 갑자기 이동평균선이나 볼린저밴드의 가격이 없어진 겁니까? 그냥...


bank_app_reviews 데이터베이스 확인/생성 완료
bank_app_reviews.viva.republica.toss_reviews 데이터 저장 완료(append)


KeyboardInterrupt: 